In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load silver layer tables
orders = spark.table("novacart_dev.silver.slv_orders")
order_items = spark.table("novacart_dev.silver.slv_order_items")
products = spark.table("novacart_dev.silver.slv_products")
customers = spark.table("novacart_dev.silver.slv_customers")
exchange = spark.table("novacart_dev.silver.slv_exchange_rates")

In [0]:
# Fact Table: Order Items (grain: one row per order line item)
fact_order_items = order_items \
    .join(orders, "order_id", "inner") \
    .join(products, "product_id", "left") \
    .join(exchange, orders.currency == exchange.currency_code, "left") \
    .select(
        # Keys
        F.col("order_item_id").alias("order_item_key"),
        F.col("order_id").alias("order_key"),
        F.col("product_id").alias("product_key"),
        F.col("customer_id").alias("customer_key"),
        F.col("order_date").alias("date_key"),
        
        # Measures
        "quantity",
        "unit_price",
        "line_total",
        F.round(
            F.col("line_total") * F.coalesce(F.col("exchange_rate_to_usd"), F.lit(1.0)),
            2
        ).alias("revenue_usd"),
        
        # Attributes (denormalized for query performance)
        "order_status_clean",
        "channel",
        orders.country_code.alias("order_country"),
        "category",
        orders.currency.alias("order_currency"),
        
        # Data Quality Flags
        F.col("dq_line_total_mismatch"),
        F.col("dq_orphan_product"),
        F.col("dq_missing_order_item_id"),
        order_items.dq_invalid_quantity,
        order_items.dq_invalid_price,
        
        # Metadata
        F.current_timestamp().alias("etl_timestamp")
    )

print(f"fact_order_items: {fact_order_items.count():,} line items")
display(fact_order_items.limit(5))

In [0]:
# Write fact_order_items to Gold layer (partitioned by date for performance)
fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date_key") \
    .saveAsTable("novacart_dev.gold.fact_order_items")

print("✓ fact_order_items written to novacart_dev.gold.fact_order_items")
print("  Partitioned by: date_key")